# Session 5: MCP Protocol & Ask IRA Architecture

In [ ]:
from src.mcp_servers.registry import MCPRegistry
from src.mcp_servers.base import MCPRequest
import asyncio

registry = MCPRegistry()
print(f"Available MCP servers: {registry.server_names}")

## 1. Individual Server Queries

In [ ]:
async def query_all():
    results = await registry.dispatch_all("Analyze AAPL stock")
    for name, response in results.items():
        print(f"\n=== {name} ===")
        print(response.content)

await query_all()

## 2. Building a Custom MCP Server

In [ ]:
from src.mcp_servers.base import MCPServer, MCPRequest, MCPResponse

class EarningsMCPServer(MCPServer):
    async def handle(self, request: MCPRequest) -> MCPResponse:
        ticker = request.query.split()[0].upper()
        data = {
            "revenue": 394_328_000_000,
            "net_income": 93_736_000_000,
            "eps": 6.04,
            "pe_ratio": 28.5,
        }
        return MCPResponse(
            content=f"{ticker} Earnings: Rev ${data['revenue']/1e9:.1f}B, EPS ${data['eps']}, P/E {data['pe_ratio']}x",
            source="earnings",
            metadata={"ticker": ticker, **data},
        )

earnings = EarningsMCPServer()
result = await earnings.handle(MCPRequest(query="AAPL earnings Q1"))
print(result.content)

## 3. Ask IRA End-to-End

In [ ]:
from src.agents.graph import run_ira

result = await run_ira("What is the investment outlook for Apple?")
print("=== Report ===")
print(result.get("report", "No report generated.")[:800])